In [ ]:
pip install sentence-transformers faiss-cpu langchain-community langchain-huggingface

In [ ]:
!pip install sentence_transformers

In [ ]:
!pip install -q --upgrade torch sentence-transformers

In [36]:
from sentence_transformers import SentenceTransformer

# 1. Carregar o modelo especialista em similaridade (STS)
# Este é o BERTimbau ajustado para entender que frases com sentidos parecidos devem ter vetores parecidos.
model_name = 'neuralmind/bert-base-portuguese-cased'
# Alternativa excelente (Serafim): 'PORTULAN/serafim-900m-portuguese-pt-sentence-encoder'

model = SentenceTransformer(model_name)

# 2. Seus dados (Exemplo: Trechos segmentados de documentos)
docs = [
    "contratação de transporte escolar para canoinhas, e a outra trata da contratação de serviços de caminhões e máquinas pesadas para o município.",
    "a prestação de serviços era feita de forma incompleta e precária"
]

# 3. Gerar os Embeddings (Transformar texto em vetor numérico)
doc_embeddings = model.encode(docs)

# Exemplo de Busca (RAG)
query = "Qual é o objeto a ser contratado pela licitação?"
query_embedding = model.encode(query)

# Comparar a pergunta com os documentos (usando similaridade de cosseno)
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity([query_embedding], doc_embeddings)[0]

# Mostrar resultados
print("Similaridade com a pergunta:")
for doc, score in zip(docs, scores):
    print(f"[{score:.4f}] {doc}")

No sentence-transformers model found with name neuralmind/bert-base-portuguese-cased. Creating a new one with mean pooling.


Similaridade com a pergunta:
[0.5821] contratação de transporte escolar para canoinhas, e a outra trata da contratação de serviços de caminhões e máquinas pesadas para o município.
[0.4817] a prestação de serviços era feita de forma incompleta e precária


In [12]:
#small set for test before run in all data

import pandas as pd
df=pd.read_csv("600_gabarito.csv")
df=df.iloc[:10]
df.to_csv('just_test.csv',index=False)

In [1]:
# -*- coding: utf-8 -*-
"""
pipeline_rag_hibrido.py

Pipeline de Extração usando RAG (Retrieval-Augmented Generation).
1. Quebra o texto da notícia em chunks.
2. Converte chunks em vetores (Embeddings) usando BERTimbau Jurídico.
3. Busca semanticamente os trechos mais relevantes para 'modalidade' e 'objeto'.
4. Envia APENAS os trechos relevantes para o LLM extrair o JSON.
"""

import pandas as pd
import json
import os
from tqdm import tqdm
from typing import List, Dict, Any

# --- Imports do RAG ---
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings # Atualizado
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

# ===========================
# CONFIGURAÇÕES
# ===========================
CAMINHO_CSV_ENTRADA = "just_test.csv"
CAMINHO_CSV_SAIDA = "resultado_rag_final.csv"

# Configuração do LLM (Ollama)
OLLAMA_HOST = "https://ollama-dev.ceos.ufsc.br"
LLM_MODEL = "gpt-oss:20b"
LLM_TEMP = 0

# Configuração dos Embeddings (O "Ímã")
# Usando o modelo jurídico do STJ que discutimos, excelente para similaridade
EMBEDDING_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
# Alternativa mais leve se ficar lento: "neuralmind/bert-base-portuguese-cased"

# ===========================
# 1. PREPARAÇÃO DOS MODELOS (Carregar apenas uma vez)
# ===========================
print("--- Carregando modelos... ---")

# A. Inicializar Modelo de Embedding (Pode demorar um pouco na 1ª vez para baixar)
print(f"Carregando modelo de embedding: {EMBEDDING_MODEL_NAME}...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

# B. Inicializar LLM
print(f"Conectando ao Ollama: {LLM_MODEL}...")
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
llm = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_HOST, temperature=LLM_TEMP)

# C. Configurar o Divisor de Texto (Chunker)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      # Tamanho de cada pedaço (caracteres)
    chunk_overlap=100,   # Sobreposição para não perder contexto nas bordas
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("--- Modelos carregados! Iniciando Pipeline RAG ---")

# ===========================
# 2. FUNÇÃO CORE DO RAG
# ===========================
def processar_noticia_com_rag(texto_noticia: str) -> Dict[str, Any]:
    """
    Recebe o texto completo da notícia e retorna o JSON extraído via RAG.
    """
    if not isinstance(texto_noticia, str) or len(texto_noticia.strip()) < 50:
        return {"objeto": [], "modalidade": []}

    # PASSO A: Segmentação (Chunking)
    chunks = text_splitter.create_documents([texto_noticia])
    
    # Se o texto for muito curto, nem precisa de RAG, manda direto (opcional)
    # Mas vamos manter o padrão para consistência.

    # PASSO B: Criação do Banco Vetorial Temporário (In-Memory)
    # Isso cria o índice pesquisável apenas para ESTA notícia
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2}) # Pega os top 2 trechos

    # PASSO C: Recuperação (Retrieval) - As Perguntas ao Texto
    
    # Query 1: Focada em Modalidade (ajuda a evitar "fraude na concorrência")
    docs_mod = retriever.invoke("Qual a modalidade legal de licitação? pregão concorrência tomada de preços leilão")
    contexto_mod = "\n".join([f"- {doc.page_content}" for doc in docs_mod])

    # Query 2: Focada em Objeto
    docs_obj = retriever.invoke("Qual o objeto da licitação? O que está sendo contratado, comprado ou construído?")
    contexto_obj = "\n".join([f"- {doc.page_content}" for doc in docs_obj])

    # PASSO D: Geração (A Chamada ao LLM)
    # Agora o prompt vê apenas os trechos filtrados, não a notícia inteira
    
    prompt = f"""
Você é um especialista em licitações. Analise os trechos recuperados abaixo para extrair as informações.

--- TRECHOS RELEVANTES SOBRE MODALIDADE ---
{contexto_mod}

--- TRECHOS RELEVANTES SOBRE OBJETO ---
{contexto_obj}

Com base APENAS nos trechos acima, extraia:
1. objeto: O que está sendo licitado.
2. modalidade: A modalidade legal (ex: Pregão, Concorrência).
   * Atenção: Se o texto falar "fraude na concorrência", ISSO NÃO É MODALIDADE.
   * Só extraia se for o tipo do certame (ex: "abriu Concorrência nº X").

Retorne JSON puro: {{ "objeto": [], "modalidade": [] }}
"""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        result = response.content.strip()
        
        # Limpeza básica de markdown
        if result.startswith("```json"): result = result[7:]
        if result.startswith("```"): result = result[3:]
        if result.endswith("```"): result = result[:-3]
        
        data = json.loads(result.strip())
        
        # Normalização simples de lista
        def normalize(val):
            if isinstance(val, str) and val: return [val]
            if isinstance(val, list): return [str(v) for v in val if v]
            return []

        return {
            "objeto": normalize(data.get("objeto")),
            "modalidade": normalize(data.get("modalidade"))
        }
        
    except Exception as e:
        # Em caso de erro no LLM ou JSON, retorna vazio
        return {"objeto": [], "modalidade": []}

# ===========================
# 3. LOOP PRINCIPAL
# ===========================

def main():
    # Carregar Dados
    try:
        df = pd.read_csv(CAMINHO_CSV_ENTRADA)
        print(f"Dataset carregado: {len(df)} linhas.")
    except FileNotFoundError:
        print("Arquivo de entrada não encontrado.")
        return

    # Preparar colunas de saída
    if 'pred_rag_objeto' not in df.columns:
        df['pred_rag_objeto'] = None
        df['pred_rag_modalidade'] = None

    # Processar (Usando tqdm para barra de progresso)
    print("Iniciando processamento RAG...")
    
    # Para teste rápido, vamos processar apenas as primeiras 10. 
    # Remova o .head(10) para processar tudo.
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        
        texto = str(row['texto_noticia'])
        
        # Executar o Pipeline RAG
        resultado = processar_noticia_com_rag(texto)
        
        # Salvar no DataFrame
        df.at[idx, 'pred_rag_objeto'] = str(resultado['objeto'])
        df.at[idx, 'pred_rag_modalidade'] = str(resultado['modalidade'])
        
        # Save parcial a cada 10
        if idx % 10 == 0:
            df.to_csv(CAMINHO_CSV_SAIDA, index=False)

    # Salvar Final
    df.to_csv(CAMINHO_CSV_SAIDA, index=False)
    print(f"Processamento concluído! Salvo em: {CAMINHO_CSV_SAIDA}")

if __name__ == "__main__":
    main()

/opt/conda/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.11/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-11-18 16:52:46.848262: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-18 16:52:46.862314: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763484766.87

--- Carregando modelos... ---
Carregando modelo de embedding: neuralmind/bert-base-portuguese-cased...


No sentence-transformers model found with name neuralmind/bert-base-portuguese-cased. Creating a new one with mean pooling.


Conectando ao Ollama: gpt-oss:20b...
--- Modelos carregados! Iniciando Pipeline RAG ---
Dataset carregado: 10 linhas.
Iniciando processamento RAG...


100%|██████████| 10/10 [02:17<00:00, 13.73s/it]

Processamento concluído! Salvo em: resultado_rag_final.csv


In [8]:
import pandas as pd
df=pd.read_csv('196_validation.csv')
df.columns

Index(['link_noticia', 'titulo', 'texto_noticia', 'municipio_ente',
       'municipio_presente', 'municipios_extraidos', 'Unnamed: 6', 'edital',
       'edital_presente', 'editais_extraidos', 'validaçao_edital',
       'Unnamed: 11', 'modalidade_licitacao', 'modalidade_presente',
       'modalidades_extraidas', 'validaçao_modalidade', 'Unnamed: 16',
       'objeto', 'objeto_presente', 'objeto_extraido', 'validaçao_objeto'],
      dtype='object')

In [11]:
df=df.drop(columns=['municipios_extraidos', 'Unnamed: 6','editais_extraidos', 'validaçao_edital','Unnamed: 11','modalidades_extraidas', 'validaçao_modalidade', 'Unnamed: 16','objeto_extraido', 'validaçao_objeto'])
df.columns

Index(['link_noticia', 'titulo', 'texto_noticia', 'municipio_ente',
       'municipio_presente', 'edital', 'edital_presente',
       'modalidade_licitacao', 'modalidade_presente', 'objeto',
       'objeto_presente'],
      dtype='object')

In [7]:
print(df.loc[0][ 'objeto'])
print('\n')
print(df.loc[0]['pred_rag_objeto'])

tinha como objeto o registro de preços para a contratação de serviços de horas-máquina, como motoniveladoras e rolos compactadores.


['registro de preços para a contratação de serviços de horas-máquina, como motoniveladoras e rolos compactadores.']


In [14]:
df['titulo_texto']=df['titulo']+' '+df['texto_noticia']
df.columns

Index(['link_noticia', 'titulo', 'texto_noticia', 'municipio_ente',
       'municipio_presente', 'edital', 'edital_presente',
       'modalidade_licitacao', 'modalidade_presente', 'objeto',
       'objeto_presente', 'titulo_texto'],
      dtype='object')

In [22]:
df.to_csv('196_validation.csv',index=False)

In [24]:
#small set for test before run in all data

import pandas as pd
df=pd.read_csv("196_validation.csv")
df=df.iloc[:5]
df[['modalidade_presente','objeto_presente']]=df[['modalidade_presente','objeto_presente']].astype('Int64')

df.to_csv('5_test.csv',index=False)

In [12]:
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

# Função para obter o embedding de uma frase
def get_embedding(sentence, model, tokenizer):
    inputs = tokenizer(sentence, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Pegando o vetor do token [CLS] como a representação da frase
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze()
    return embedding

# Função para calcular a similaridade de cosseno entre dois embeddings
def calculate_similarity(embedding1, embedding2):
    similarity = cosine_similarity(embedding1.unsqueeze(0), embedding2.unsqueeze(0))
    return similarity[0][0]

# Definir o nome do modelo
EMBEDDING_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

# Carregar o modelo e o tokenizador
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
model = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME)

# Frases a serem comparadas
sentence1 = "tinha como objeto o registro de preços para a contratação de serviços de horas-máquina, como motoniveladoras e rolos compactadores."
sentence2 = 'registro de preços para a contratação de serviços de horas-máquina, como motoniveladoras e rolos compactadores.'

# Obter os embeddings das frases
embedding1 = get_embedding(sentence1, model, tokenizer)
embedding2 = get_embedding(sentence2, model, tokenizer)

# Calcular a similaridade
similarity = calculate_similarity(embedding1, embedding2)

print(f"Similaridade entre as frases: {similarity:.4f}")


Similaridade entre as frases: 0.9511


PIPELINE FINAL SIMILARIDADE SEMÂNTICA + RAG

In [31]:
# -*- coding: utf-8 -*-
"""
pipeline_rag_avaliacao_integrado.py

Pipeline integrado: RAG + Avaliação com Similaridade Semântica
- Usa RAG para extrair atributos (municipio, modalidade, edital, objeto)
- Avalia contra ground truth usando:
  * Match exato para: edital, modalidade, municipio
  * Similaridade semântica para: objeto
"""

import pandas as pd
import json
import os
from tqdm import tqdm
from typing import List, Dict, Any, Optional, Tuple
from sklearn.metrics.pairwise import cosine_similarity

# --- RAG Imports ---
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

# ===========================
# CONFIGURAÇÕES
# ===========================
CAMINHO_CSV_ENTRADA = "196_validation.csv"
CAMINHO_CSV_PARCIAL = "resultado_rag_parcial.csv"
CAMINHO_CSV_FINAL = "resultado_rag_final.csv"

# Ground Truth Columns
COLUNAS_GT = {
    'titulo_texto': 'titulo_texto',
    'municipio': {'valor': 'municipio_ente', 'presente': 'municipio_presente'},
    'modalidade': {'valor': 'modalidade_licitacao', 'presente': 'modalidade_presente'},
    'edital': {'valor': 'edital', 'presente': 'edital_presente'},
    'objeto': {'valor': 'objeto', 'presente': 'objeto_presente'}
}


# Atributos a processar
AVALIAR = {
    'municipio': True,
    'modalidade': True,
    'edital': True,
    'objeto': True
}

# Estratégia de Métrica por Atributo
METRICAS_POR_ATRIBUTO = {
    'edital': 'exact',
    'modalidade': 'exact',
    'municipio': 'exact',
    'objeto': 'semantic'
}

SIMILARITY_THRESHOLD = 0.83  # Para atributos semânticos

# Configuração LLM
OLLAMA_HOST = "https://ollama-dev.ceos.ufsc.br"
LLM_MODEL = "gpt-oss:20b"
LLM_TEMP = 0

# Configuração Embeddings
EMBEDDING_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

# Parâmetros de Execução
START_INDEX = 0
END_INDEX = -1  # -1 = processar até o fim
SAVE_EVERY = 20
RESUME_PROCESSING = True

# ===========================
# 1. PREPARAÇÃO (Carregar uma vez)
# ===========================
print("--- Carregando modelos... ---")

print(f"Carregando embeddings: {EMBEDDING_MODEL_NAME}...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

print(f"Conectando ao Ollama: {LLM_MODEL}...")
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
llm = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_HOST, temperature=LLM_TEMP)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("--- Modelos carregados! ---\n")

# ===========================
# 2. FUNÇÕES AUXILIARES
# ===========================
def clean_gt_value(value: Any) -> Optional[str]:
    """Limpa valores do ground truth."""
    if pd.isna(value) or value is None:
        return None
    val_str = str(value).strip()
    if val_str in ["", "[]", "0", "-"]:
        return None
    return val_str

def normalize_edital(edital_str: Optional[str]) -> Optional[str]:
    """Normaliza número de edital (remove zeros à esquerda)."""
    if edital_str is None:
        return None
    parts = str(edital_str).strip().split('/')
    if len(parts) == 2:
        num_norm = parts[0].lstrip('0') or '0'
        return f"{num_norm}/{parts[1]}"
    return str(edital_str).strip()

# ===========================
# 3. FUNÇÃO RAG
# ===========================
def processar_noticia_com_rag(texto_noticia: str) -> Dict[str, Any]:
    """Extrai atributos usando RAG."""
    if not isinstance(texto_noticia, str) or len(texto_noticia.strip()) < 50:
        return {"objeto": [], "modalidade": [], "edital": [], "municipio": []}

    # A. Chunking
    chunks = text_splitter.create_documents([texto_noticia])
    
    # B. Indexação (banco vetorial temporário)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

    # C. Retrieval (queries especializadas por atributo)
    docs_mun = retriever.invoke("Qual o município do edital de licitação?")
    contexto_mun = "\n".join([f"- {doc.page_content}" for doc in docs_mun])
    
    docs_mod = retriever.invoke("Qual a modalidade legal de licitação? pregão concorrência tomada de preços")
    contexto_mod = "\n".join([f"- {doc.page_content}" for doc in docs_mod])
    
    docs_edi = retriever.invoke("Qual o número do edital de licitação?")
    contexto_edi = "\n".join([f"- {doc.page_content}" for doc in docs_edi])
    
    docs_obj = retriever.invoke("Qual o objeto da licitação? O que está sendo contratado?")
    contexto_obj = "\n".join([f"- {doc.page_content}" for doc in docs_obj])

    # D. Generation (Prompt ao LLM)
    prompt = f"""
Você é um especialista em licitações. Analise os trechos recuperados abaixo.

--- MUNICÍPIO ---
{contexto_mun}

--- MODALIDADE ---
{contexto_mod}

--- EDITAL ---
{contexto_edi}

--- OBJETO ---
{contexto_obj}

Extraia APENAS com base nos trechos acima:
1. municipio: Município do edital (se houver vários igualmente importantes: "vários municípios*")
2. modalidade: Modalidade legal (ex: Pregão, Concorrência). IGNORE "fraude na concorrência".
3. edital: Número do edital (formato: "123/2023")
4. objeto: O que está sendo licitado (texto descritivo)

Retorne JSON puro:
{{
  "municipio": [],
  "modalidade": [],
  "edital": [],
  "objeto": []
}}
"""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        result = response.content.strip()
        
        # Limpeza de markdown
        if result.startswith("```json"): result = result[7:]
        if result.startswith("```"): result = result[3:]
        if result.endswith("```"): result = result[:-3]
        
        data = json.loads(result.strip())
        
        # Normalização
        def normalize(val):
            if isinstance(val, str) and val: return [val]
            if isinstance(val, list): return [str(v) for v in val if v]
            return []

        resultado = {
            "municipio": normalize(data.get("municipio")),
            "modalidade": normalize(data.get("modalidade")),
            "edital": [normalize_edital(e) for e in normalize(data.get("edital")) if e],
            "objeto": normalize(data.get("objeto"))
        }
        
        return resultado
        
    except Exception as e:
        return {"objeto": [], "modalidade": [], "edital": [], "municipio": []}

# ===========================
# 4. CLASSIFICAÇÃO (Exact + Semantic)
# ===========================
def classificar_predicao(gt_val: Optional[str], pred_list: List[str], attr: str) -> Tuple[str, Optional[float]]:
    """
    Classifica predição usando estratégia apropriada.
    Retorna: (classificacao, similarity_score)
    """
    has_gt = gt_val is not None
    has_pred = bool(pred_list)
    
    # Casos triviais
    if not has_gt and not has_pred:
        return "TN", None
    if not has_gt and has_pred:
        return "FP", 0.0
    if has_gt and not has_pred:
        return "FN", 0.0
    
    # Escolher estratégia
    strategy = METRICAS_POR_ATRIBUTO.get(attr, 'exact')
    
    if strategy == 'exact':
        # Match exato normalizado
        if attr == 'edital':
            gt_set = {normalize_edital(item.strip()) for item in str(gt_val).split(',') if item.strip()}
            pred_set = {normalize_edital(p) for p in pred_list if str(p).strip()}
        else:
            gt_set = {item.strip().lower() for item in str(gt_val).split(',') if item.strip()}
            pred_set = {str(p).strip().lower() for p in pred_list if str(p).strip()}
        
        is_match = (gt_set == pred_set)
        return ("TP" if is_match else "FP"), None
    
    elif strategy == 'semantic':
        # Similaridade de cosseno
        gt_text = str(gt_val).strip()
        pred_text = ' '.join(str(p) for p in pred_list).strip()
        
        if not pred_text:
            return "FN", 0.0
        
        # Calcular embeddings
        gt_emb = embeddings.embed_query(gt_text)
        pred_emb = embeddings.embed_query(pred_text)
        
        # Similaridade
        similarity = cosine_similarity([gt_emb], [pred_emb])[0][0]
        
        # Classificar baseado no threshold
        if similarity >= SIMILARITY_THRESHOLD:
            return "TP", float(similarity)
        else:
            return "FP", float(similarity)
    
    return "FP", None

# ===========================
# 5. CÁLCULO DE MÉTRICAS
# ===========================
def calculate_metrics(tp: int, fp: int, fn: int, tn: int) -> Dict[str, float]:
    """Calcula Precision, Recall, F1, Accuracy."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    return {
        'accuracy': accuracy, 'precision': precision, 
        'recall': recall, 'f1_score': f1,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
    }

def print_metrics_table(tabela: Dict):
    """Imprime tabela de métricas."""
    print("="*100)
    print(f"| {'Atributo':<15} | {'Acc':<7} | {'Prec':<7} | {'Rec':<7} | {'F1':<7} | {'TP':<4} | {'FP':<4} | {'FN':<4} | {'TN':<4} |")
    print("="*100)
    for attr, metrics in tabela.items():
        if 'status' in metrics:
            print(f"| {attr:<15} | {metrics['status']:<60} |")
        else:
            print(f"| {attr:<15} | {metrics['accuracy']:<7.2%} | {metrics['precision']:<7.2%} | {metrics['recall']:<7.2%} | {metrics['f1_score']:<7.2%} | {int(metrics['tp']):<4} | {int(metrics['fp']):<4} | {int(metrics['fn']):<4} | {int(metrics['tn']):<4} |")
    print("="*100)

# ===========================
# 6. PIPELINE PRINCIPAL
# ===========================
def main():
    print("\n=== PIPELINE RAG + AVALIAÇÃO INTEGRADO ===\n")
    
    # Carregar CSV
    try:
        df = pd.read_csv(CAMINHO_CSV_ENTRADA)
        print(f"Dataset carregado: {len(df)} linhas\n")
    except FileNotFoundError:
        print(f"ERRO: Arquivo '{CAMINHO_CSV_ENTRADA}' não encontrado.")
        return

    # Configurar slice de processamento
    total_rows = len(df)
    start = max(0, START_INDEX)
    end = total_rows if END_INDEX == -1 else min(END_INDEX, total_rows)
    
    if start >= end:
        print(f"START_INDEX ({start}) >= END_INDEX ({end}). Nada a processar.")
        return
    
    num_to_process = end - start
    print(f"Processando linhas {start} a {end-1} ({num_to_process} itens)\n")
    
    # Preparar colunas de saída
    attributes_enabled = [attr for attr, enabled in AVALIAR.items() if enabled]
    for attr in attributes_enabled:
        for col in [f'gt_raw_{attr}', f'gt_efetivo_{attr}', f'pred_{attr}', 
                    f'classif_{attr}', f'similarity_{attr}']:
            if col not in df.columns:
                df[col] = None
    
    # Configurar resume
    marker_col = f'classif_{attributes_enabled[0]}' if attributes_enabled else None
    is_resuming = RESUME_PROCESSING and marker_col and marker_col in df.columns
    
    # PROCESSAR NOTÍCIAS
    print("Iniciando processamento...\n")
    
    for idx, row in tqdm(df.iloc[start:end].iterrows(), total=num_to_process, desc="Processando"):
        
        # Pular se já processado
        if is_resuming and pd.notna(row.get(marker_col)):
            continue
        
        texto = str(row[COLUNAS_GT['titulo_texto']])
        
        # Executar RAG
        preds = processar_noticia_com_rag(texto)
        
        # Avaliar cada atributo
        for attr in attributes_enabled:
            col_config = COLUNAS_GT[attr]
            valor_gt_bruto = row.get(col_config['valor'])
            presente = str(row.get(col_config['presente'], '')).strip() in ['1', '1.0']
            
            gt_val = clean_gt_value(valor_gt_bruto) if presente else None
            pred_list = preds.get(attr, [])
            
            # Classificar
            classif, similarity = classificar_predicao(gt_val, pred_list, attr)
            
            # Salvar no DataFrame
            df.at[idx, f'gt_raw_{attr}'] = valor_gt_bruto
            df.at[idx, f'gt_efetivo_{attr}'] = gt_val
            df.at[idx, f'pred_{attr}'] = str(pred_list)
            df.at[idx, f'classif_{attr}'] = classif
            df.at[idx, f'similarity_{attr}'] = similarity
        
        # Salvamento parcial
        loop_count = (idx - start) + 1
        if loop_count % SAVE_EVERY == 0:
            try:
                df.to_csv(CAMINHO_CSV_PARCIAL, index=False, encoding='utf-8-sig')
                tqdm.write(f"Backup salvo em '{CAMINHO_CSV_PARCIAL}' (índice {idx})")
            except Exception as e:
                tqdm.write(f"Erro ao salvar backup: {e}")
    
    # AGREGAÇÃO DE MÉTRICAS
    print("\nAgregando métricas...\n")
    
    df_slice = df.iloc[start:end]
    tabela = {}
    
    for attr in attributes_enabled:
        classif_col = f'classif_{attr}'
        if classif_col not in df_slice.columns:
            tabela[attr] = {'status': 'NÃO PROCESSADO'}
            continue
        
        valid_rows = df_slice[classif_col].dropna()
        if len(valid_rows) == 0:
            tabela[attr] = {'status': 'SEM DADOS'}
            continue
        
        counts = valid_rows.value_counts()
        tp = counts.get('TP', 0)
        fp = counts.get('FP', 0)
        fn = counts.get('FN', 0)
        tn = counts.get('TN', 0)
        
        tabela[attr] = calculate_metrics(tp, fp, fn, tn)
    
    # SALVAR CSV FINAL
    print(f"Salvando resultado final em '{CAMINHO_CSV_FINAL}'...\n")
    try:
        # Ordenar colunas
        final_cols = ['texto_noticia']
        for attr in AVALIAR.keys():
            final_cols.extend([c for c in df.columns if c.endswith(f'_{attr}')])
        final_cols.extend([c for c in df.columns if c not in final_cols])
        
        df_export = df[[c for c in final_cols if c in df.columns]]
        df_export.to_csv(CAMINHO_CSV_FINAL, index=False, encoding='utf-8-sig')
        print(f" Arquivo final salvo: '{CAMINHO_CSV_FINAL}'")
    except Exception as e:
        print(f" Erro ao salvar CSV final: {e}")
        df.to_csv(f"BACKUP_{CAMINHO_CSV_FINAL}", index=False, encoding='utf-8-sig')
    
    # IMPRIMIR RESULTADOS
    print("\n" + "="*100)
    print("                         RESULTADOS DA AVALIAÇÃO")
    print(f"                    (Slice: linhas {start} a {end-1})")
    print("="*100 + "\n")
    print_metrics_table(tabela)
    
    # Média geral
    accuracies = [m['accuracy'] for m in tabela.values() if 'accuracy' in m]
    if accuracies:
        avg_acc = sum(accuracies) / len(accuracies)
        print(f"\nACURÁCIA MÉDIA GERAL: {avg_acc:.2%}\n")
    
    print("="*100)
    print("✓ Processamento concluído!")
    print("="*100 + "\n")

if __name__ == "__main__":
    main()

--- Carregando modelos... ---
Carregando embeddings: neuralmind/bert-base-portuguese-cased...


No sentence-transformers model found with name neuralmind/bert-base-portuguese-cased. Creating a new one with mean pooling.


Conectando ao Ollama: gpt-oss:20b...
--- Modelos carregados! ---


=== PIPELINE RAG + AVALIAÇÃO INTEGRADO ===

Dataset carregado: 196 linhas

Processando linhas 0 a 195 (196 itens)

Iniciando processamento...



Processando:  10%|█         | 20/196 [02:28<21:21,  7.28s/it] 

Backup salvo em 'resultado_rag_parcial.csv' (índice 19)


...

Backup salvo em 'resultado_rag_parcial.csv' (índice 159)


Processando:  92%|█████████▏| 180/196 [18:33<01:25,  5.32s/it]

Backup salvo em 'resultado_rag_parcial.csv' (índice 179)


Processando: 100%|██████████| 196/196 [20:06<00:00,  6.15s/it]


Agregando métricas...

Salvando resultado final em 'resultado_rag_final.csv'...

 Arquivo final salvo: 'resultado_rag_final.csv'

                         RESULTADOS DA AVALIAÇÃO
                    (Slice: linhas 0 a 195)

| Atributo        | Acc     | Prec    | Rec     | F1      | TP   | FP   | FN   | TN   |
| municipio       | 58.16%  | 65.55%  | 65.55%  | 65.55%  | 78   | 41   | 41   | 36   |
| modalidade      | 80.61%  | 51.02%  | 64.10%  | 56.82%  | 25   | 24   | 14   | 133  |
| edital          | 96.43%  | 75.00%  | 69.23%  | 72.00%  | 9    | 3    | 4    | 180  |
| objeto          | 37.24%  | 38.24%  | 57.14%  | 45.81%  | 52   | 84   | 39   | 21   |

ACURÁCIA MÉDIA GERAL: 68.11%

✓ Processamento concluído!

